In [110]:
from pathlib import Path
import os
import sys
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Shared configuration
BLOCK_SIZE = 64
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10

# print("Python:", sys.executable)
# print("PyTorch:", torch.__version__)
# print("CUDA build:", torch.version.cuda)
# print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


Loading Tokenizer

In [10]:
PROJECT_DIR = Path.cwd()
nested_project_dir = PROJECT_DIR / "Deep_Learning" / "Transformer_Language_model"
if (nested_project_dir / "data" / "raw").is_dir():
    PROJECT_DIR = nested_project_dir

while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "data" / "raw").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "data" / "raw").is_dir():
    raise FileNotFoundError("Could not locate the Transformer_Language_model project directory")

TOKENIZER_DIR = PROJECT_DIR / "models" / "krish_bpe"

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
MERGES_PATH = TOKENIZER_DIR / "merges.txt"

assert VOCAB_PATH.exists(), f"Missing: {VOCAB_PATH}"
assert MERGES_PATH.exists(), f"Missing: {MERGES_PATH}"

tokenizer = ByteLevelBPETokenizer(
    str(VOCAB_PATH),
    str(MERGES_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()
EOS_ID = tokenizer.token_to_id("<|eos|>")

assert EOS_ID is not None, "EOS token not found!"

print("Tokenizer loaded successfully!")
print("Vocabulary size:", VOCAB_SIZE)
print("EOS ID:", EOS_ID)

Tokenizer loaded successfully!
Vocabulary size: 2000
EOS ID: 3


In [11]:
girl_df = pd.read_csv(PROJECT_DIR / "data" / "raw" / "girl_only.csv")

# Use the actual column names from your CSV

TEXT_COLUMN = "text"

messages = (
    girl_df[TEXT_COLUMN]
    .dropna()
    .astype(str)
)

messages = messages[
    messages.str.strip() != ""
].tolist()

print("Total text messages:", len(messages))

print("\nFirst 5 messages:")
for message in messages[:5]:
    print(message)

Total text messages: 55623

First 5 messages:
Koi baat nhi
Tumne msg kya kiya tha?
Happy Republic Day 😊
Btw
Same to u


Dividing the Messages into Train , validation and test split 

In [12]:
total_messages = len(messages)

train_end = int(total_messages * 0.90)
val_end = int(total_messages * 0.95)

train_messages = messages[:train_end]
val_messages = messages[train_end:val_end]
test_messages = messages[val_end:]

print("Training messages:", len(train_messages))
print("Validation messages:", len(val_messages))
print("Testing messages:", len(test_messages))
print("Total:", len(train_messages) + len(val_messages) + len(test_messages))

Training messages: 50060
Validation messages: 2781
Testing messages: 2782
Total: 55623


In [13]:
def encode_messages(messages, tokenizer):

    all_ids = []

    eos_id = tokenizer.token_to_id("<|eos|>")

    for message in messages:

        token_ids = tokenizer.encode(message).ids

        all_ids.extend(token_ids)
        all_ids.append(eos_id)

    return torch.tensor(all_ids, dtype=torch.long)


train_ids = encode_messages(train_messages, tokenizer)
val_ids = encode_messages(val_messages, tokenizer)
test_ids = encode_messages(test_messages, tokenizer)

print("Training token IDs:", len(train_ids))
print("Validation token IDs:", len(val_ids))
print("Testing token IDs:", len(test_ids))

print("\nFirst 20 training IDs:")
print(train_ids[:20].tolist())

Training token IDs: 391531
Validation token IDs: 19513
Testing token IDs: 19832

First 20 training IDs:
[760, 346, 286, 3, 1598, 289, 617, 347, 503, 343, 34, 3, 446, 862, 92, 224, 1909, 83, 1363, 391]


In [14]:
BLOCK_SIZE = 64

class NextTokenDataset(Dataset):

    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):

        x = self.token_ids[idx : idx + self.block_size]

        y = self.token_ids[idx + 1 : idx + self.block_size + 1]

        return x, y


train_dataset = NextTokenDataset(train_ids, BLOCK_SIZE)
val_dataset = NextTokenDataset(val_ids, BLOCK_SIZE)
test_dataset = NextTokenDataset(test_ids, BLOCK_SIZE)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Testing sequences:", len(test_dataset))

Training sequences: 391467
Validation sequences: 19449
Testing sequences: 19768


DataLoaders

In [15]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 12234
Validation batches: 608
Testing batches: 618


In [16]:
x, y = next(iter(train_loader))

print("Input shape:", x.shape)
print("Target shape:", y.shape)

print("\nFirst input sequence:")
print(x[0].tolist())

print("\nFirst target sequence:")
print(y[0].tolist())

print("\nDecoded input:")
print(tokenizer.decode(x[0].tolist()))

print("\nDecoded target:")
print(tokenizer.decode(y[0].tolist()))

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])

First input sequence:
[298, 583, 3, 664, 710, 284, 294, 984, 422, 79, 913, 492, 1228, 1871, 3, 1882, 1111, 1087, 379, 314, 1337, 322, 290, 80, 3, 429, 287, 1201, 383, 78, 283, 303, 1834, 268, 3, 648, 268, 3, 999, 523, 475, 1991, 288, 377, 287, 401, 1192, 3, 382, 554, 3, 55, 68, 372, 326, 492, 525, 917, 1055, 288, 434, 88, 1235, 3]

First target sequence:
[583, 3, 664, 710, 284, 294, 984, 422, 79, 913, 492, 1228, 1871, 3, 1882, 1111, 1087, 379, 314, 1337, 322, 290, 80, 3, 429, 287, 1201, 383, 78, 283, 303, 1834, 268, 3, 648, 268, 3, 999, 523, 475, 1991, 288, 377, 287, 401, 1192, 3, 382, 554, 3, 55, 68, 372, 326, 492, 525, 917, 1055, 288, 434, 88, 1235, 3, 278]

Decoded input:
 ho jaye<|eos|>Abhi itna dur travel karega tujhe jarurat padegi<|eos|>kabhi alwida na kehna sanam<|eos|>abhi to mulakat bhi baaki hai<|eos|>Thik hai<|eos|>Jab recharge kar le to bata diyo<|eos|>Achha sun<|eos|>Taki mai tujhe call karke pareshan k

Bigram model 

In [17]:

# # ============================================================
# # 1. BUILD BIGRAM TRANSITION COUNTS
# # ============================================================

# import torch
# import torch.nn.functional as F
# import matplotlib.pyplot as plt
# import math
# from pathlib import Path

# # Add smoothing to handle unseen token transitions
# ALPHA = 0.1

# # Each cell [i, j] stores how often token j follows token i
# transition_counts = torch.zeros(
#     (VOCAB_SIZE, VOCAB_SIZE),
#     dtype=torch.float64
# )

# # Count adjacent token pairs from training data
# current_tokens = train_ids[:-1]
# next_tokens = train_ids[1:]

# pair_indices = (
#     current_tokens * VOCAB_SIZE + next_tokens
# )

# counts = torch.bincount(
#     pair_indices,
#     minlength=VOCAB_SIZE * VOCAB_SIZE
# )

# transition_counts = counts.reshape(
#     VOCAB_SIZE, VOCAB_SIZE
# ).double()

# print("Transition counts created!")
# print("Matrix shape:", transition_counts.shape)
# print("Total transitions:", int(transition_counts.sum()))


# # ============================================================
# # 2. CALCULATE NEXT-TOKEN PROBABILITIES
# # ============================================================

# smoothed_counts = transition_counts + ALPHA

# transition_probs = (
#     smoothed_counts /
#     smoothed_counts.sum(dim=1, keepdim=True)
# )

# # Store log probabilities for efficient evaluation
# log_probs = torch.log(transition_probs).to(device)

# print("Probability matrix shape:", transition_probs.shape)
# print("Probability matrix ready!")


# # ============================================================
# # 3. EVALUATION FUNCTION
# # ============================================================

# def evaluate_bigram(token_ids):

#     token_ids = token_ids.to(device)

#     x = token_ids[:-1]
#     y = token_ids[1:]

#     # Select probability of the actual next token
#     selected_log_probs = log_probs[x, y]

#     loss = -selected_log_probs.mean()

#     # Most likely next-token prediction
#     predictions = log_probs[x].argmax(dim=1)

#     accuracy = (predictions == y).float().mean()

#     perplexity = torch.exp(loss)

#     return {
#         "loss": loss.item(),
#         "accuracy": accuracy.item(),
#         "perplexity": perplexity.item()
#     }


# # ============================================================
# # 4. VALIDATION
# # ============================================================

# val_results = evaluate_bigram(val_ids)

# print("\nVALIDATION RESULTS")
# print("----------------------")
# print("Loss:", round(val_results["loss"], 4))
# print("Accuracy:", round(val_results["accuracy"], 4))
# print("Perplexity:", round(val_results["perplexity"], 4))


# # ============================================================
# # 5. FINAL TEST EVALUATION
# # ============================================================

# test_results = evaluate_bigram(test_ids)

# print("\nTEST RESULTS")
# print("----------------------")
# print("Loss:", round(test_results["loss"], 4))
# print("Accuracy:", round(test_results["accuracy"], 4))
# print("Perplexity:", round(test_results["perplexity"], 4))


# # ============================================================
# # 6. GENERATE TEXT
# # ============================================================

# def generate_bigram(
#     prompt,
#     max_new_tokens=50,
#     temperature=1.0
# ):

#     if temperature <= 0:
#         raise ValueError("Temperature must be positive.")

#     model_probs = transition_probs

#     encoded = tokenizer.encode(prompt).ids

#     if not encoded:
#         raise ValueError("Prompt cannot be empty.")

#     generated = encoded.copy()

#     for _ in range(max_new_tokens):

#         current_token = generated[-1]

#         probabilities = model_probs[current_token]

#         # Temperature sampling
#         probabilities = torch.softmax(
#             torch.log(probabilities) / temperature,
#             dim=0
#         )

#         next_token = torch.multinomial(
#             probabilities.float(),
#             num_samples=1
#         ).item()

#         if next_token == EOS_ID:
#             break

#         generated.append(next_token)

#     return tokenizer.decode(generated)


# # ============================================================
# # 7. GENERATE SAMPLE TEXT
# # ============================================================

# sample = generate_bigram(
#     prompt="I",
#     max_new_tokens=50,
#     temperature=0.8
# )

# print("\nGENERATED TEXT")
# print("----------------------")
# print(sample)


# # ============================================================
# # 8. SAVE BIGRAM MODEL
# # ============================================================

# SAVE_DIR = Path("models/trained")
# SAVE_DIR.mkdir(parents=True, exist_ok=True)

# torch.save(
#     {
#         "transition_counts": transition_counts,
#         "transition_probs": transition_probs,
#         "vocab_size": VOCAB_SIZE,
#         "alpha": ALPHA,
#         "test_results": test_results
#     },
#     SAVE_DIR / "bigram_model.pth"
# )

# print("\nBigram model saved successfully!")


# # ============================================================
# # 9. VISUALIZE MOST LIKELY NEXT TOKENS
# # ============================================================

# def show_next_token_predictions(token, top_k=10):

#     token_ids = tokenizer.encode(token).ids

#     if not token_ids:
#         print("Token not recognized.")
#         return

#     current_id = token_ids[-1]

#     probabilities = transition_probs[current_id]

#     top_probs, top_ids = torch.topk(
#         probabilities,
#         k=top_k
#     )

#     print(f"\nMost likely tokens after: {token}")

#     for token_id, probability in zip(
#         top_ids.tolist(),
#         top_probs.tolist()
#     ):

#         decoded = tokenizer.decode([token_id])

#         print(
#             repr(decoded),
#             "Probability:",
#             round(probability, 4)
#         )


# show_next_token_predictions("I", top_k=10)


# print("\nBIGRAM EXPERIMENT COMPLETED!")


## LSTM

In [18]:
### This is the old code before overfitting occurs


# class LSTMLanguageModel(nn.Module):

#     def __init__(
#         self,
#         vocab_size,
#         embedding_dim=128,
#         hidden_dim=128,
#         num_layers=1
#     ):
#         super().__init__()

#         # Convert token IDs into embedding vectors
#         self.embedding = nn.Embedding(
#             vocab_size,
#             embedding_dim
#         )

#         # LSTM processes sequential information
#         self.lstm = nn.LSTM(
#             input_size=embedding_dim,
#             hidden_size=hidden_dim,
#             num_layers=num_layers,
#             batch_first=True
#         )

#         # Convert LSTM outputs into vocabulary logits
#         self.fc = nn.Linear(
#             hidden_dim,
#             vocab_size
#         )

#     def forward(self, x):

#         x = self.embedding(x)

#         output, (hidden_state, cell_state) = self.lstm(x)

#         logits = self.fc(output)

#         return logits





class LSTMLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=1,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):

        x = self.embedding(x)

        x = self.embedding_dropout(x)

        output, (hidden_state, cell_state) = self.lstm(x)

        output = self.output_dropout(output)

        logits = self.fc(output)

        return logits

In [19]:
# EMBEDDING_DIM = 128
# HIDDEN_DIM = 128
# NUM_LAYERS = 1

# lstm_model = LSTMLanguageModel(
#     vocab_size=VOCAB_SIZE,
#     embedding_dim=EMBEDDING_DIM,
#     hidden_dim=HIDDEN_DIM,
#     num_layers=NUM_LAYERS
# ).to(device)

# print(lstm_model)

# total_params = sum(
#     parameter.numel()
#     for parameter in lstm_model.parameters()
# )

# print("\nTotal parameters:", f"{total_params:,}")
# print("Device:", device)



EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.2

lstm_model = LSTMLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    lstm_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

GRAD_CLIP = 1.0
EPOCHS = 5
PATIENCE = 2

print("Model parameters:", sum(
    p.numel() for p in lstm_model.parameters()
))

print("Device:", device)
print("Maximum epochs:", EPOCHS)
print("Dropout:", DROPOUT)
print("Weight decay:", 0.0001)

Model parameters: 646096
Device: cuda
Maximum epochs: 5
Dropout: 0.2
Weight decay: 0.0001


In [20]:
print("Model device:", next(lstm_model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated GPU memory:",
        round(torch.cuda.memory_allocated() / 1024**2, 2),
        "MB"
    )

Model device: cuda:0
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Allocated GPU memory: 2.46 MB


In [21]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    lstm_model.parameters(),
    lr=0.001
)

GRAD_CLIP = 1.0
EPOCHS = 5



Training 

In [22]:
# def train_one_epoch(model, loader, optimizer, criterion):

#     model.train()

#     total_loss = 0
#     total_correct = 0
#     total_tokens = 0

#     for x, y in loader:

#         x = x.to(device)
#         y = y.to(device)

#         # Reset previous gradients
#         optimizer.zero_grad()

#         # Forward pass
#         logits = model(x)

#         # Calculate loss
#         loss = criterion(
#             logits.reshape(-1, VOCAB_SIZE),
#             y.reshape(-1)
#         )

#         # Backpropagation
#         loss.backward()

#         # Prevent exploding gradients
#         nn.utils.clip_grad_norm_(
#             model.parameters(),
#             GRAD_CLIP
#         )

#         # Update parameters
#         optimizer.step()

#         # Calculate accuracy
#         predictions = logits.argmax(dim=-1)

#         total_correct += (predictions == y).sum().item()
#         total_tokens += y.numel()
#         total_loss += loss.item() * y.numel()

#     avg_loss = total_loss / total_tokens
#     accuracy = total_correct / total_tokens

#     return avg_loss, accuracy

In [23]:
def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )

            predictions = logits.argmax(dim=-1)

            total_correct += (predictions == y).sum().item()
            total_tokens += y.numel()
            total_loss += loss.item() * y.numel()

    avg_loss = total_loss / total_tokens
    accuracy = total_correct / total_tokens

    return avg_loss, accuracy

In [24]:
# # history = {
    
# #     "train_loss": [],
# #     "val_loss": [],
# #     "train_accuracy": [],
# #     "val_accuracy": []
# # }

# # start_time = time.time()

# # for epoch in range(EPOCHS):

# #     train_loss, train_acc = train_one_epoch(
# #         lstm_model,
# #         train_loader,
# #         optimizer,
# #         criterion
# #     )

# #     val_loss, val_acc = evaluate_model(
# #         lstm_model,
# #         val_loader,
# #         criterion
# #     )

# #     history["train_loss"].append(train_loss)
# #     history["val_loss"].append(val_loss)
# #     history["train_accuracy"].append(train_acc)
# #     history["val_accuracy"].append(val_acc)

# #     print(
# #         f"Epoch [{epoch + 1}/{EPOCHS}] "
# #         f"| Train Loss: {train_loss:.4f} "
# #         f"| Val Loss: {val_loss:.4f} "
# #         f"| Train Accuracy: {train_acc:.4f} "
# #         f"| Val Accuracy: {val_acc:.4f}"
# #     )

# # training_time = time.time() - start_time

# # print("\nTraining time:", round(training_time, 2), "seconds")

# import time
# import copy
# import os

# history = {
#     "train_loss": [],
#     "val_loss": [],
#     "train_accuracy": [],
#     "val_accuracy": []
# }

# best_val_loss = float("inf")
# best_model_state = None
# epochs_without_improvement = 0
# best_epoch = 0

# os.makedirs("models/trained", exist_ok=True)

# start_time = time.time()

# for epoch in range(EPOCHS):

#     train_loss, train_acc = train_one_epoch(
#         lstm_model,
#         train_loader,
#         optimizer,
#         criterion
#     )

#     val_loss, val_acc = evaluate_model(
#         lstm_model,
#         val_loader,
#         criterion
#     )

#     history["train_loss"].append(train_loss)
#     history["val_loss"].append(val_loss)
#     history["train_accuracy"].append(train_acc)
#     history["val_accuracy"].append(val_acc)

#     print(
#         f"Epoch [{epoch + 1}/{EPOCHS}] "
#         f"| Train Loss: {train_loss:.4f} "
#         f"| Val Loss: {val_loss:.4f} "
#         f"| Train Accuracy: {train_acc:.4f} "
#         f"| Val Accuracy: {val_acc:.4f}"
#     )

#     # Check whether validation loss improved
#     if val_loss < best_val_loss - 0.0001:

#         best_val_loss = val_loss
#         best_epoch = epoch + 1

#         best_model_state = copy.deepcopy(
#             lstm_model.state_dict()
#         )

#         epochs_without_improvement = 0

#         torch.save(
#             best_model_state,
#             "models/trained/lstm_best.pth"
#         )

#         print("  Best model saved!")

#     else:
#         epochs_without_improvement += 1

#     # Early stopping
#     if epochs_without_improvement >= PATIENCE:

#         print("\nEarly stopping triggered!")
#         break

# # Restore best model
# if best_model_state is not None:
#     lstm_model.load_state_dict(best_model_state)

# training_time = time.time() - start_time

# print("\nTraining completed.")
# print("Best epoch:", best_epoch)
# print("Best validation loss:", round(best_val_loss, 4))
# print("Training time:", round(training_time, 2), "seconds")

In [25]:
import matplotlib.pyplot as plt

if "history" in globals() and history.get("train_loss"):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="Training Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("LSTM Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_accuracy"], label="Training Accuracy")
    plt.plot(history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("LSTM Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("LSTM training was skipped; no history to plot.")


LSTM training was skipped; no history to plot.


The model was overfiited in the first try so we added dropout etc.

Generating text using our Trained Model 

In [26]:
def generate_text(
    model,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=40
):

    model.eval()

    # Convert prompt into token IDs
    input_ids = tokenizer.encode(prompt).ids

    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )

    generated_ids = input_ids[0].tolist()

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Keep only the most recent context
            context = input_ids[:, -BLOCK_SIZE:]

            # Get predictions
            logits = model(context)

            # Last position's predictions
            logits = logits[:, -1, :]

            # Control randomness
            logits = logits / temperature

            # Keep only top-k predictions
            values, indices = torch.topk(logits, top_k)

            filtered_logits = torch.full_like(
                logits, float("-inf")
            )

            filtered_logits.scatter_(1, indices, values)

            # Convert scores into probabilities
            probabilities = torch.softmax(
                filtered_logits, dim=-1
            )

            # Sample next token
            next_token = torch.multinomial(
                probabilities, num_samples=1
            )

            token_id = next_token.item()

            # Stop at end-of-message token
            if token_id == EOS_ID:
                break

            generated_ids.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    return tokenizer.decode(generated_ids)

In [27]:
prompt = "I love"

generated_text = generate_text(
    lstm_model,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=40
)

print("Generated text:")
print(generated_text)

Generated text:
I love karta padhdumter padhnemoitniDvi tujhse11Vhi chal akeleSGood gir payegimeetly sh hoteIt dilChal� liya lag�hb pai maine akele usme❤️~ kharab� kartaTo rehte pade lagikaEk thankide cr peBhai Tu karleokk dhu hona aayegi farCall wct mujhko hoga msgs 🥲 mor bre abVhi yhahee morningmmmm aas ladke papa�matlabjhalo sahiee� akele vajah patibha walhmmmm jakr� khataDo sochtission� �


In [28]:
prompts = [
    "I am going",
    "Today I feel",
    "Do you know"
]

for prompt in prompts:

    print("\nPrompt:", prompt)

    print(
        generate_text(
            lstm_model,
            prompt,
            max_new_tokens=50,
            temperature=0.8,
            top_k=40
        )
    )

    print("-" * 50)


Prompt: I am going
I am going joce gar mas mujhe naa dekhnanga minseekhMaCo� papa problemSo rehtikisCo edi bhu 😁😁 dekhtiust rahungiTujheHai😘 dino dekhnamm itood samne seriMa padtahehehehe!kis:// pre hone passMere paungi
--------------------------------------------------

Prompt: Today I feel
Today I feel <hut sari suna� beyque� sh�❤️yy bhool�Di ladkecti11Chup😁 starNahi khar preya aaneVaise paisa�legeekhnhi�diaaaaaaaaaaaaaaaaShayd mo� dinochak�Ja dekokkBhaummy manaummpt
--------------------------------------------------

Prompt: Do you know
Do you know tujhkoVhi classwal layaIsliye padta paisetingare late kholoaldi padega it "��plssmaag dinoquech busy batheoredum bey joBha padh ladke paisa yhh jan-�� chizyaanush problem mujhkoeriod dosto paisa res
--------------------------------------------------


## Transformers 

In [109]:
# mixed_df = pd.read_csv(PROJECT_DIR / "data" / "raw" / "mixed.csv")

# print("Dataset shape:", mixed_df.shape)

# print("\nColumns:")
# print(mixed_df.columns.tolist())

# print("\nFirst 10 messages:")
# print(mixed_df[["speaker", "text"]].head(10))

# print("\nSpeaker distribution:")
# print(mixed_df["speaker"].value_counts())

# print("\nMissing values:")
# print(mixed_df.isna().sum())


In [ ]:
# # Remove rows with missing text
# mixed_df = mixed_df.dropna(subset=["text"]).copy()

# # Convert text to string and remove surrounding whitespace
# mixed_df["text"] = mixed_df["text"].astype(str).str.strip()

# # Remove empty messages
# mixed_df = mixed_df[mixed_df["text"] != ""].copy()

# # Remove non-conversational placeholders
# non_text_messages = [
#     "You deleted this message",
#     "<Media omitted>"
# ]

# mixed_df = mixed_df[
#     ~mixed_df["text"].isin(non_text_messages)
# ].copy()

# # Replace private speaker names with generic labels
# mixed_df["speaker"] = mixed_df["speaker"].replace({
#     "A***": "BOY",
#     "H*****": "GIRL"
# })

# print("Cleaned dataset shape:", mixed_df.shape)

# print("\nSpeaker distribution:")
# print(mixed_df["speaker"].value_counts())

# print("\nFirst 10 cleaned messages:")
# print(mixed_df[["speaker", "text"]].head(10))

Cleaned dataset shape: (118171, 6)

Speaker distribution:
speaker
BOY     62812
GIRL    55359
Name: count, dtype: int64

First 10 cleaned messages:
   speaker                                    text
2      BOY  Sorry sorry galti se call chla gya tha
3     GIRL                            Koi baat nhi
4     GIRL                 Tumne msg kya kiya tha?
5     GIRL                    Happy Republic Day 😊
6     GIRL                                     Btw
8      BOY                    Happy republic day 🫡
9     GIRL                               Same to u
10     BOY                      School nhi gyi kya
11    GIRL                           Aaj jana tha?
12    GIRL                            Ohh bhul gyi


In [31]:
# Check existing tokenizer

print("Vocabulary size:", tokenizer.get_vocab_size())

print("\nSpecial token IDs:")

special_tokens = [
    "<|unk|>",
    "<|pad|>",
    "<|bos|>",
    "<|eos|>"
]

for token in special_tokens:
    print(token, ":", tokenizer.token_to_id(token))

print("\nSpeaker token check:")

print("BOY:", tokenizer.token_to_id("<|boy|>"))
print("GIRL:", tokenizer.token_to_id("<|girl|>"))

Vocabulary size: 2000

Special token IDs:
<|unk|> : 0
<|pad|> : 1
<|bos|> : 2
<|eos|> : 3

Speaker token check:
BOY: None
GIRL: None


In [32]:
total_messages = len(mixed_df)

train_end = int(total_messages * 0.90)
val_end = int(total_messages * 0.95)

train_df = mixed_df.iloc[:train_end].copy()
val_df = mixed_df.iloc[train_end:val_end].copy()
test_df = mixed_df.iloc[val_end:].copy()

print("Training messages:", len(train_df))
print("Validation messages:", len(val_df))
print("Testing messages:", len(test_df))

print("\nTraining date range:")
print(train_df["timestamp_normalized"].iloc[0])
print(train_df["timestamp_normalized"].iloc[-1])

print("\nValidation date range:")
print(val_df["timestamp_normalized"].iloc[0])
print(val_df["timestamp_normalized"].iloc[-1])

Training messages: 106353
Validation messages: 5909
Testing messages: 5909

Training date range:
2025-01-26 12:29
2026-08-22 21:14

Validation date range:
2026-08-22 21:14
2026-09-07 20:14


In [33]:
import os
from tokenizers import ByteLevelBPETokenizer

MIXED_TOKENIZER_DIR = PROJECT_DIR / "models" / "mixed_bpe"
MIXED_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

mixed_tokenizer = ByteLevelBPETokenizer()

special_tokens = [
    "<|unk|>",
    "<|pad|>",
    "<|bos|>",
    "<|eos|>",
    "<|boy|>",
    "<|girl|>"
]

mixed_tokenizer.train_from_iterator(
    train_df["text"].tolist(),
    vocab_size=4000,
    min_frequency=2,
    special_tokens=special_tokens
)

mixed_tokenizer.save_model(str(MIXED_TOKENIZER_DIR))

print("Tokenizer trained and saved!")
print("Vocabulary size:", mixed_tokenizer.get_vocab_size())

print("\nSpecial token IDs:")
for token in special_tokens:
    print(token, ":", mixed_tokenizer.token_to_id(token))


Tokenizer trained and saved!
Vocabulary size: 4000

Special token IDs:
<|unk|> : 0
<|pad|> : 1
<|bos|> : 2
<|eos|> : 3
<|boy|> : 4
<|girl|> : 5


In [34]:
# Get special token IDs
BOY_ID = mixed_tokenizer.token_to_id("<|boy|>")
GIRL_ID = mixed_tokenizer.token_to_id("<|girl|>")
EOS_ID = mixed_tokenizer.token_to_id("<|eos|>")

PAD_ID = mixed_tokenizer.token_to_id("<|pad|>")
BOS_ID = mixed_tokenizer.token_to_id("<|bos|>")

def encode_conversation(df, tokenizer):

    token_ids = []

    for _, row in df.iterrows():

        # Add speaker token
        if row["speaker"] == "BOY":
            token_ids.append(BOY_ID)

        elif row["speaker"] == "GIRL":
            token_ids.append(GIRL_ID)

        # Encode message text
        message_tokens = tokenizer.encode(
            row["text"]
        ).ids

        token_ids.extend(message_tokens)

        # Mark end of message
        token_ids.append(EOS_ID)

    return token_ids


# Encode all three splits independently
train_ids = encode_conversation(train_df, mixed_tokenizer)
val_ids = encode_conversation(val_df, mixed_tokenizer)
test_ids = encode_conversation(test_df, mixed_tokenizer)

print("Training tokens:", len(train_ids))
print("Validation tokens:", len(val_ids))
print("Testing tokens:", len(test_ids))

print("\nFirst 30 training token IDs:")
print(train_ids[:30])

print("\nDecoded sample:")
print(mixed_tokenizer.decode(train_ids[:30]))


Training tokens: 812369
Validation tokens: 37721
Testing tokens: 39662

First 30 training token IDs:
[4, 583, 666, 1268, 314, 513, 1273, 439, 326, 3, 5, 650, 356, 289, 3, 5, 2067, 294, 842, 351, 537, 326, 36, 3, 5, 2453, 226, 1420, 85, 3388]

Decoded sample:
Sorry sorry galti se call chla gya thaKoi baat nhiTumne msg kya kiya tha?Happy Republic


In [35]:
print("BOY token ID:", BOY_ID)
print("GIRL token ID:", GIRL_ID)
print("EOS token ID:", EOS_ID)

print("\nFirst 20 encoded tokens:")
print(train_ids[:20])

print("\nDecoded with special tokens:")

print(
    mixed_tokenizer.decode(
        train_ids[:30],
        skip_special_tokens=False
    )
)

print("\nSpeaker token counts:")

print("BOY:", train_ids.count(BOY_ID))
print("GIRL:", train_ids.count(GIRL_ID))
print("EOS:", train_ids.count(EOS_ID))

BOY token ID: 4
GIRL token ID: 5
EOS token ID: 3

First 20 encoded tokens:
[4, 583, 666, 1268, 314, 513, 1273, 439, 326, 3, 5, 650, 356, 289, 3, 5, 2067, 294, 842, 351]

Decoded with special tokens:
<|boy|>Sorry sorry galti se call chla gya tha<|eos|><|girl|>Koi baat nhi<|eos|><|girl|>Tumne msg kya kiya tha?<|eos|><|girl|>Happy Republic

Speaker token counts:
BOY: 56351
GIRL: 50002
EOS: 106353


DATASET

In [36]:
import torch
from torch.utils.data import Dataset, DataLoader

BLOCK_SIZE = 64
STRIDE = 4
BATCH_SIZE = 32

class NextTokenDataset(Dataset):

    def __init__(self, token_ids, block_size, stride=1):

        self.token_ids = torch.tensor(
            token_ids,
            dtype=torch.long
        )

        self.block_size = block_size
        self.stride = stride

        self.length = (
            len(self.token_ids) - block_size - 1
        ) // stride + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):

        start = idx * self.stride
        end = start + self.block_size

        x = self.token_ids[start:end]
        y = self.token_ids[start + 1:end + 1]

        return x, y


train_dataset = NextTokenDataset(
    train_ids, BLOCK_SIZE, STRIDE
)

val_dataset = NextTokenDataset(
    val_ids, BLOCK_SIZE, STRIDE
)

test_dataset = NextTokenDataset(
    test_ids, BLOCK_SIZE, STRIDE
)

print("Training windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
print("Testing windows:", len(test_dataset))

x, y = train_dataset[0]

print("\nInput shape:", x.shape)
print("Target shape:", y.shape)

print("\nInput tokens:", x[:10].tolist())
print("Target tokens:", y[:10].tolist())

Training windows: 203077
Validation windows: 9415
Testing windows: 9900

Input shape: torch.Size([64])
Target shape: torch.Size([64])

Input tokens: [4, 583, 666, 1268, 314, 513, 1273, 439, 326, 3]
Target tokens: [583, 666, 1268, 314, 513, 1273, 439, 326, 3, 5]


In [37]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

x, y = next(iter(train_loader))

print("\nInput shape:", x.shape)
print("Target shape:", y.shape)

Training batches: 6347
Validation batches: 295
Testing batches: 310

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])


Transformer Configuration 


In [38]:
# Vocabulary configuration
VOCAB_SIZE = mixed_tokenizer.get_vocab_size()

PAD_ID = mixed_tokenizer.token_to_id("<|pad|>")
BOS_ID = mixed_tokenizer.token_to_id("<|bos|>")
EOS_ID = mixed_tokenizer.token_to_id("<|eos|>")
BOY_ID = mixed_tokenizer.token_to_id("<|boy|>")
GIRL_ID = mixed_tokenizer.token_to_id("<|girl|>")

# Transformer configuration
BLOCK_SIZE = 64

EMBEDDING_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.2

# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.0001
EPOCHS = 10
PATIENCE = 2
GRAD_CLIP = 1.0

print("Vocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Attention heads:", NUM_HEADS)
print("Transformer layers:", NUM_LAYERS)
print("Dropout:", DROPOUT)
print("Device:", device)

Vocabulary size: 4000
Embedding dimension: 128
Attention heads: 4
Transformer layers: 2
Dropout: 0.2
Device: cuda


In [39]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):

        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        seq_length = x.size(1)

        # Prevent each position from seeing future tokens
        causal_mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        attention_output, attention_weights = self.attention(
            x,
            x,
            x,
            attn_mask=causal_mask,
            need_weights=False
        )

        return self.dropout(attention_output)

In [40]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim, dropout):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(embedding_dim, embedding_dim * 4),

            nn.GELU(),

            nn.Linear(embedding_dim * 4, embedding_dim),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.network(x)

In [41]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):

        super().__init__()

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.attention = MultiHeadSelfAttention(
            embedding_dim,
            num_heads,
            dropout
        )

        self.norm2 = nn.LayerNorm(embedding_dim)

        self.feed_forward = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):

        # Self-attention + residual connection
        x = x + self.attention(self.norm1(x))

        # Feed-forward + residual connection
        x = x + self.feed_forward(self.norm2(x))

        return x

In [42]:
class TransformerLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size,
        dropout
    ):
        super().__init__()

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Positional embeddings
        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        self.embedding_dropout = nn.Dropout(dropout)

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        # Final normalization
        self.final_norm = nn.LayerNorm(embedding_dim)

        # Vocabulary prediction layer
        self.output_layer = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):

        batch_size, seq_length = x.shape

        # Token embeddings
        token_embeddings = self.token_embedding(x)

        # Position embeddings
        positions = torch.arange(
            seq_length,
            device=x.device
        )

        position_embeddings = self.position_embedding(positions)

        # Combine token and position information
        x = token_embeddings + position_embeddings

        x = self.embedding_dropout(x)

        # Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.final_norm(x)

        # Predict vocabulary logits
        logits = self.output_layer(x)

        return logits

In [43]:
transformer_model = TransformerLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT
).to(device)

total_params = sum(
    p.numel()
    for p in transformer_model.parameters()
)

print(transformer_model)

print("\nTotal parameters:", f"{total_params:,}")
print("Device:", next(transformer_model.parameters()).device)

TransformerLanguageModel(
  (token_embedding): Embedding(4000, 128)
  (position_embedding): Embedding(64, 128)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (blocks): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadSelfAttention(
        (attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (feed_forward): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (1): TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_af

In [44]:
# Get one batch
x, y = next(iter(train_loader))

# Move batch to GPU
x = x.to(device)
y = y.to(device)

# Test forward pass
transformer_model.eval()

with torch.no_grad():
    logits = transformer_model(x)

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Output logits shape:", logits.shape)

print("\nExpected output shape:")
print((BATCH_SIZE, BLOCK_SIZE, VOCAB_SIZE))

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])
Output logits shape: torch.Size([32, 64, 4000])

Expected output shape:
(32, 64, 4000)


Loss Function and Optimizer 

In [45]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    transformer_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

GRAD_CLIP = 1.0
EPOCHS = 5
PATIENCE = 2

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)

Loss function: CrossEntropyLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.0001
)
Learning rate: 0.0003
Weight decay: 0.0001
Maximum epochs: 5
Early stopping patience: 2


Training 

In [46]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    total_correct = 0
    total_tokens = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        optimizer.step()

        predictions = logits.argmax(dim=-1)

        total_correct += (predictions == y).sum().item()
        total_tokens += y.numel()
        total_loss += loss.item() * y.numel()

    avg_loss = total_loss / total_tokens
    accuracy = total_correct / total_tokens

    return avg_loss, accuracy

In [47]:
def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )

            predictions = logits.argmax(dim=-1)

            total_correct += (predictions == y).sum().item()
            total_tokens += y.numel()
            total_loss += loss.item() * y.numel()

    avg_loss = total_loss / total_tokens
    accuracy = total_correct / total_tokens

    return avg_loss, accuracy

In [48]:
import copy
import time

checkpoint_path = PROJECT_DIR / "models" / "trained" / "transformer_best.pth"
history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

if checkpoint_path.exists():
    print(f"Existing checkpoint found at {checkpoint_path}; skipping training.")
else:
    best_val_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0
    best_epoch = 0
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    start_time = time.time()

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            transformer_model, train_loader, optimizer, criterion
        )
        val_loss, val_acc = evaluate_model(
            transformer_model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Train Accuracy: {train_acc:.4f} "
            f"| Val Accuracy: {val_acc:.4f}"
        )

        if val_loss < best_val_loss - 0.0001:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(transformer_model.state_dict())
            epochs_without_improvement = 0
            torch.save(best_model_state, checkpoint_path)
            print("Best model saved!")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping triggered!")
            break

    if best_model_state is None:
        raise RuntimeError("Training completed without producing a model checkpoint.")

    transformer_model.load_state_dict(best_model_state)
    print("\nTraining completed!")
    print("Best epoch:", best_epoch)
    print("Best validation loss:", round(best_val_loss, 4))
    print("Training time:", round(time.time() - start_time, 2), "seconds")


Existing checkpoint found at d:\guar dai\AI-ML-Projects\Deep_Learning\Transformer_Language_model\models\trained\transformer_best.pth; skipping training.


In [49]:
import matplotlib.pyplot as plt

if history["train_loss"]:
    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history["train_loss"], marker="o", label="Training Loss")
    plt.plot(epochs_range, history["val_loss"], marker="o", label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Transformer Loss")
    plt.xticks(list(epochs_range))
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history["train_accuracy"], marker="o", label="Training Accuracy")
    plt.plot(epochs_range, history["val_accuracy"], marker="o", label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Transformer Accuracy")
    plt.xticks(list(epochs_range))
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No new training history to plot; using the saved checkpoint.")


No new training history to plot; using the saved checkpoint.


Best Transformer Checkpoint 

In [50]:
checkpoint_path = PROJECT_DIR / "models" / "trained" / "transformer_best.pth"

transformer_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

transformer_model.eval()

print("Best Transformer checkpoint loaded!")


Best Transformer checkpoint loaded!


Testing our model

In [51]:
test_loss, test_accuracy = evaluate_model(
    transformer_model,
    test_loader,
    criterion
)

test_perplexity = math.exp(test_loss)

print("\n========== TRANSFORMER TEST RESULTS ==========")

print(f"Test Loss       : {test_loss:.4f}")
print(f"Test Accuracy   : {test_accuracy:.4%}")
print(f"Test Perplexity : {test_perplexity:.2f}")

print("==============================================")


========== TRANSFORMER TEST RESULTS ==========
Test Loss       : 3.8468
Test Accuracy   : 29.6062%
Test Perplexity : 46.84


Girl's side response generation

In [52]:
def generate_girl_response(
    model,
    conversation_history,
    max_new_tokens=50,
    temperature=0.8,
    top_k=40
):

    model.eval()

    # Encode conversation history
    input_ids = []

    for speaker, message in conversation_history:

        if speaker == "BOY":
            input_ids.append(BOY_ID)

        elif speaker == "GIRL":
            input_ids.append(GIRL_ID)

        message_ids = mixed_tokenizer.encode(message).ids

        input_ids.extend(message_ids)
        input_ids.append(EOS_ID)

    # Add GIRL marker to request her response
    input_ids.append(GIRL_ID)

    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )

    generated_ids = []

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Restrict context to model's maximum block size
            context = input_ids[:, -BLOCK_SIZE:]

            logits = model(context)

            # Get predictions for the final position
            logits = logits[:, -1, :]

            # Apply temperature
            logits = logits / temperature

            # Prevent generating speaker markers
            logits[:, BOY_ID] = float("-inf")
            logits[:, GIRL_ID] = float("-inf")
            logits[:, PAD_ID] = float("-inf")
            logits[:, BOS_ID] = float("-inf")

            # Top-k sampling
            values, indices = torch.topk(logits, top_k)

            filtered_logits = torch.full_like(
                logits, float("-inf")
            )

            filtered_logits.scatter_(1, indices, values)

            probabilities = torch.softmax(
                filtered_logits, dim=-1
            )

            next_token = torch.multinomial(
                probabilities, num_samples=1
            )

            token_id = next_token.item()

            # Stop at end of message
            if token_id == EOS_ID:
                break

            generated_ids.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    return mixed_tokenizer.decode(generated_ids)

In [53]:
conversation = [
    ("BOY", "Kaisi ho? "),
]

response = generate_girl_response(
    transformer_model,
    conversation,
    max_new_tokens=50,
    temperature=0.8,
    top_k=40
)

print("BOY: Kaisi ho?")
print("GIRL:", response)

BOY: Kaisi ho?
GIRL: Mai bas itna bura lag rha tha


In [77]:
conversation = [
    ("BOY", "Kaisi ho?"),
    ("GIRL", "Thik hoon, tum batao?"),
    ("BOY", "Main bhi thik hoon."),
    ("GIRL", "Aaj kya kiya?"),
    ("BOY", "College gaya tha."),
    ("GIRL", "Achha, kaisa tha?"),
    ("BOY", "Bahut boring tha.")
]

response = generate_girl_response(
    transformer_model,
    conversation,
    max_new_tokens=50,
    temperature=0.8,
    top_k=40
)

print("GIRL:", response)

GIRL: Achha


Starting the process of deployment 

Packaging the artifacts 

In [78]:
from pathlib import Path
import shutil
import json

# Deployment directories
DEPLOY_DIR = PROJECT_DIR / "deployment"
ARTIFACT_DIR = DEPLOY_DIR / "artifacts"
TOKENIZER_DEPLOY_DIR = ARTIFACT_DIR / "tokenizer"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

# Source paths
checkpoint_src = PROJECT_DIR / "models" / "trained" / "transformer_best.pth"
vocab_src = PROJECT_DIR / "models" / "mixed_bpe" / "vocab.json"
merges_src = PROJECT_DIR / "models" / "mixed_bpe" / "merges.txt"

# Verify required files
for path in [checkpoint_src, vocab_src, merges_src]:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")

# Copy artifacts
shutil.copy2(checkpoint_src, ARTIFACT_DIR / "transformer_best.pth")
shutil.copy2(vocab_src, TOKENIZER_DEPLOY_DIR / "vocab.json")
shutil.copy2(merges_src, TOKENIZER_DEPLOY_DIR / "merges.txt")

# Save exact model configuration
config = {
    "vocab_size": VOCAB_SIZE,
    "block_size": BLOCK_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "pad_id": PAD_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID,
    "boy_id": BOY_ID,
    "girl_id": GIRL_ID
}

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Deployment artifacts packaged successfully!")

for path in DEPLOY_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

Deployment artifacts packaged successfully!
deployment\artifacts\config.json
deployment\artifacts\transformer_best.pth
deployment\artifacts\tokenizer\merges.txt
deployment\artifacts\tokenizer\vocab.json


In [79]:
model_code = r'''
import json
from pathlib import Path

import torch
import torch.nn as nn
from tokenizers import ByteLevelBPETokenizer


BASE_DIR = Path(__file__).resolve().parent
ARTIFACT_DIR = BASE_DIR / "artifacts"
TOKENIZER_DIR = ARTIFACT_DIR / "tokenizer"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -------------------------
# Attention
# -------------------------

class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_length = x.size(1)

        causal_mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        attention_output, _ = self.attention(
            x, x, x,
            attn_mask=causal_mask,
            need_weights=False
        )

        return self.dropout(attention_output)


# -------------------------
# Feed Forward
# -------------------------

class FeedForward(nn.Module):

    def __init__(self, embedding_dim, dropout):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


# -------------------------
# Transformer Block
# -------------------------

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(embedding_dim)

        self.attention = MultiHeadSelfAttention(
            embedding_dim,
            num_heads,
            dropout
        )

        self.norm2 = nn.LayerNorm(embedding_dim)

        self.feed_forward = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


# -------------------------
# Transformer Language Model
# -------------------------

class TransformerLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size,
        dropout
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(embedding_dim)

        self.output_layer = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):
        _, seq_length = x.shape

        token_embeddings = self.token_embedding(x)

        positions = torch.arange(
            seq_length,
            device=x.device
        )

        position_embeddings = self.position_embedding(positions)

        x = token_embeddings + position_embeddings
        x = self.embedding_dropout(x)
        x = self.blocks(x)
        x = self.final_norm(x)

        return self.output_layer(x)


# -------------------------
# Load model and tokenizer
# -------------------------

with open(ARTIFACT_DIR / "config.json", "r") as f:
    CONFIG = json.load(f)

tokenizer = ByteLevelBPETokenizer(
    str(TOKENIZER_DIR / "vocab.json"),
    str(TOKENIZER_DIR / "merges.txt")
)

model = TransformerLanguageModel(
    vocab_size=CONFIG["vocab_size"],
    embedding_dim=CONFIG["embedding_dim"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

checkpoint = torch.load(
    ARTIFACT_DIR / "transformer_best.pth",
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint)
model.eval()


# Special tokens
PAD_ID = CONFIG["pad_id"]
BOS_ID = CONFIG["bos_id"]
EOS_ID = CONFIG["eos_id"]
BOY_ID = CONFIG["boy_id"]
GIRL_ID = CONFIG["girl_id"]


# -------------------------
# Girl response generation
# -------------------------

def generate_girl_response(
    conversation_history,
    max_new_tokens=50,
    temperature=0.8,
    top_k=40
):

    if temperature <= 0:
        raise ValueError("Temperature must be positive.")

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    model.eval()

    input_ids = []

    for speaker, message in conversation_history:

        if speaker == "BOY":
            input_ids.append(BOY_ID)

        elif speaker == "GIRL":
            input_ids.append(GIRL_ID)

        else:
            raise ValueError(f"Unknown speaker: {speaker}")

        input_ids.extend(tokenizer.encode(message).ids)
        input_ids.append(EOS_ID)

    # Request a girl response
    input_ids.append(GIRL_ID)

    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated_ids = []

    with torch.no_grad():

        for _ in range(max_new_tokens):

            context = input_ids[:, -CONFIG["block_size"]:]

            logits = model(context)
            logits = logits[:, -1, :]

            logits = logits / temperature

            # Block special tokens that should not appear in replies
            for token_id in [BOY_ID, GIRL_ID, PAD_ID, BOS_ID]:
                logits[:, token_id] = float("-inf")

            current_top_k = min(top_k, logits.size(-1))

            values, indices = torch.topk(
                logits,
                current_top_k
            )

            filtered_logits = torch.full_like(
                logits,
                float("-inf")
            )

            filtered_logits.scatter_(1, indices, values)

            probabilities = torch.softmax(
                filtered_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            token_id = next_token.item()

            if token_id == EOS_ID:
                break

            generated_ids.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    return tokenizer.decode(generated_ids)
'''

model_file = DEPLOY_DIR / "model.py"
model_file.write_text(model_code, encoding="utf-8")

print("model.py created successfully!")
print("Location:", model_file)
print("Size:", model_file.stat().st_size, "bytes")

model.py created successfully!
Location: d:\guar dai\AI-ML-Projects\Deep_Learning\Transformer_Language_model\deployment\model.py
Size: 7234 bytes


In [82]:
import sys
import importlib

sys.path.insert(0, str(DEPLOY_DIR))

import model

print("Model module imported successfully!")
print("Device:", model.DEVICE)

Model module imported successfully!
Device: cuda


In [108]:
conversation = [
    ("BOY", "Tm thk ho ?")
]

response = model.generate_girl_response(
    conversation_history=conversation,
    max_new_tokens=50,
    temperature=0.8,
    top_k=40
)

print("Generated GIRL response:")
print(response)

Generated GIRL response:
Yaar
